# MetroGNN — Training Template

**Architecture:** Full Metropolitan Graph → GraphSAGE Encoder → Subgraph Pooling → Multi-task Heads → Structured JSON → LLM Layer

```
┌────────────────────────────┐
│ Full Metropolitan Graph    │
│ (OSM + GTFS + SIRI + etc.) │
└────────────┬───────────────┘
             ↓
┌────────────────────────────┐
│ Node / Edge Feature Builder│
└────────────┬───────────────┘
             ↓
┌────────────────────────────┐
│  GraphSAGE Encoder (×3)    │
└────────────┬───────────────┘
             ↓  node embeddings
┌────────────────────────────┐
│  Subgraph Pooling (mean)   │
└────────────┬───────────────┘
             ↓  area embedding vector
   ┌─────────┬─────────┬─────────┬─────────┬─────────┐
   ↓         ↓         ↓         ↓         ↓         ↓
Access.  Connect.  Transit   Bike    Network   Area
  Reg.     Reg.     Reg.     Reg.    Role Cls  Type Cls
             ↓
┌────────────────────────────┐
│  Structured JSON Output    │
└────────────┬───────────────┘
             ↓
┌────────────────────────────┐
│  LLM Explanation Layer     │
└────────────────────────────┘
```

---
**Standalone — no dependency on the Map Chat backend.**

---
## 0. Setup & Config

In [ ]:
# pip install torch torch_geometric osmnx geopandas pandas numpy scikit-learn umap-learn matplotlib

import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv, BatchNorm, global_mean_pool
from torch_geometric.data import Data, DataLoader
from torch_geometric.utils import k_hop_subgraph

import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import json
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    city          = 'Tel Aviv-Yafo, Israel',
    proximity_km  = 3.0,      # edge between nodes within this radius (fallback)
    k_hops        = 3,        # GNN layers = k-hop receptive field
    subgraph_r_m  = 1500,     # inference: subgraph radius in metres
    hidden_dim    = 128,
    n_role_cls    = 4,        # isolated / residential / local_hub / metro_hub
    n_area_cls    = 5,        # residential / commercial / mixed / transit / industrial
    lr            = 1e-3,
    epochs        = 100,
    batch_size    = 32,
    dropout       = 0.2,
    data_dir      = Path('data'),
    out_dir       = Path('outputs'),
)
CFG['data_dir'].mkdir(exist_ok=True)
CFG['out_dir'].mkdir(exist_ok=True)

ROLE_LABELS = ['isolated', 'residential', 'local_hub', 'metropolitan_hub']
AREA_LABELS = ['residential', 'commercial', 'mixed_use', 'transit_hub', 'industrial']

---
## 1. Graph Construction

Node = real road intersection | Edge = road segment

Sources:
- **OSM** — road network via `osmnx`
- **GTFS** — bus/rail stops (Israel MoT feed)
- **SIRI** — real-time delay feed (stub)
- **Micro-mobility** — trip start/end counts (stub)

In [ ]:
# ── 1A. OSM road network ─────────────────────────────────────────────────────
print('Downloading OSM road network...')
G_osm = ox.graph_from_place(CFG['city'], network_type='drive')
G_osm = ox.add_edge_speeds(G_osm)
G_osm = ox.add_edge_travel_times(G_osm)

print(f'OSM nodes: {G_osm.number_of_nodes():,}  edges: {G_osm.number_of_edges():,}')

# Save for reuse
ox.save_graphml(G_osm, CFG['data_dir'] / 'osm_road.graphml')
# ox.load_graphml(CFG['data_dir'] / 'osm_road.graphml')  # reload without re-download

In [ ]:
# ── 1B. GTFS stops ───────────────────────────────────────────────────────────
# Download Israel MoT GTFS: https://gtfs.mot.gov.il/
# Expected columns after parsing: stop_id, stop_name, stop_lat, stop_lon, route_count, avg_headway_min

GTFS_PATH = CFG['data_dir'] / 'stops_enriched.csv'

if GTFS_PATH.exists():
    stops_df = pd.read_csv(GTFS_PATH)
    print(f'Loaded {len(stops_df):,} GTFS stops')
else:
    # ── Stub: synthetic stops near OSM nodes ─────────────────────────────────
    print('GTFS not found — generating synthetic stubs')
    rng = np.random.default_rng(42)
    node_ids = list(G_osm.nodes)
    n_stops = min(500, len(node_ids))
    chosen = rng.choice(node_ids, n_stops, replace=False)
    stops_df = pd.DataFrame({
        'stop_id'        : chosen,
        'stop_name'      : [f'Stop_{i}' for i in range(n_stops)],
        'stop_lat'       : [G_osm.nodes[n]['y'] for n in chosen],
        'stop_lon'       : [G_osm.nodes[n]['x'] for n in chosen],
        'route_count'    : rng.integers(1, 15, n_stops),
        'avg_headway_min': rng.uniform(5, 30, n_stops).round(1),
        'avg_delay_min'  : rng.uniform(0, 8, n_stops).round(1),
    })
    stops_df.to_csv(GTFS_PATH, index=False)

stops_df.head(3)

In [ ]:
# ── 1C. SIRI real-time delay feed (stub) ─────────────────────────────────────
# Replace with actual SIRI API call when available.
# Expected schema: stop_id → avg_delay_min (rolling 30-day average)

siri_delays = dict(zip(stops_df['stop_id'], stops_df['avg_delay_min']))
print(f'SIRI delay entries: {len(siri_delays):,}')

In [ ]:
# ── 1D. Micro-mobility trips (stub) ──────────────────────────────────────────
# Replace with real Bird/Lime/HaBalam trip data.
# Expected schema: osm_node_id → {starts, ends}

rng = np.random.default_rng(0)
node_ids_list = list(G_osm.nodes)
micro_starts = dict(zip(
    rng.choice(node_ids_list, 300, replace=False),
    rng.integers(0, 200, 300)
))
micro_ends = dict(zip(
    rng.choice(node_ids_list, 300, replace=False),
    rng.integers(0, 200, 300)
))
print(f'Micro-mobility anchors: starts={len(micro_starts)}  ends={len(micro_ends)}')

---
## 2. Node & Edge Feature Builder

### Node features (14)
| Feature | Source |
|---|---|
| degree | OSM graph |
| intersection_density | OSM (nodes/km²) |
| road_type | OSM highway tag |
| bus_stops_nearby | GTFS |
| unique_bus_lines | GTFS |
| avg_headway | GTFS |
| avg_delay | SIRI |
| micromobility_starts | micro-mobility data |
| micromobility_ends | micro-mobility data |
| bike_lane_length | OSM cycleway tag |
| simulated_activity_flow | simulator (stub) |
| distance_to_lrt | OSM/GTFS |
| distance_to_train | OSM/GTFS |
| landuse_mix | OSM landuse |

### Edge features (8)
| Feature | Source |
|---|---|
| road_length | OSM |
| travel_time | OSM + speed |
| speed_limit | OSM |
| lanes | OSM |
| bike_lane | OSM cycleway |
| bus_route_exists | GTFS |
| simulated_flow | simulator (stub) |
| congestion | SIRI / stub |

In [ ]:
from sklearn.neighbors import BallTree
from math import radians

# Pre-index stops for fast spatial lookup
stop_coords_rad = np.deg2rad(stops_df[['stop_lat', 'stop_lon']].values)
stop_tree = BallTree(stop_coords_rad, metric='haversine')

ROAD_TYPE_MAP = {
    'motorway': 0, 'trunk': 1, 'primary': 2, 'secondary': 3,
    'tertiary': 4, 'residential': 5, 'service': 6, 'other': 7
}


def _safe(val, default=0.0):
    try:
        v = float(val[0] if isinstance(val, list) else val)
        return v if np.isfinite(v) else default
    except Exception:
        return default


def build_node_features(G):
    """Returns (node_id_list, feature_matrix N×14)"""
    node_ids = list(G.nodes)
    rows = []
    for nid in node_ids:
        nd = G.nodes[nid]
        lat, lon = nd.get('y', 0.0), nd.get('x', 0.0)

        # degree
        deg = G.degree(nid)

        # intersection_density: nodes within 500m
        pt = np.deg2rad([[lat, lon]])
        cnt = stop_tree.query_radius(pt, r=500 / 6371, count_only=True)[0]
        density = cnt / (np.pi * 0.5 ** 2)  # per km²

        # road_type from incident edges
        rt_vals = [G.edges[nid, nb, 0].get('highway', 'other') for nb in G.successors(nid)]
        rt_str = rt_vals[0] if rt_vals else 'other'
        if isinstance(rt_str, list): rt_str = rt_str[0]
        road_type = ROAD_TYPE_MAP.get(rt_str, 7) / 7.0  # normalised

        # GTFS: stops within 300m
        idxs = stop_tree.query_radius(pt, r=300 / 6371)[0]
        nearby_stops = stops_df.iloc[idxs]
        bus_stops_nearby = len(nearby_stops)
        unique_lines     = nearby_stops['route_count'].sum() if len(nearby_stops) else 0
        avg_headway      = nearby_stops['avg_headway_min'].mean() if len(nearby_stops) else 30.0
        avg_delay        = nearby_stops['avg_delay_min'].mean()   if len(nearby_stops) else 0.0

        # Micro-mobility
        mm_starts = micro_starts.get(nid, 0)
        mm_ends   = micro_ends.get(nid, 0)

        # Bike lane length from incident edges
        bike_len = sum(
            _safe(G.edges[nid, nb, 0].get('length', 0))
            for nb in G.successors(nid)
            if G.edges[nid, nb, 0].get('cycleway') not in (None, 'no')
        )

        # Simulated activity flow (stub)
        sim_flow = float(deg) * 10.0  # placeholder

        # Distance to LRT / train (stub: use nearest stop with route_count > 5)
        heavy_stops = stops_df[stops_df['route_count'] > 5]
        if len(heavy_stops):
            h_coords = np.deg2rad(heavy_stops[['stop_lat', 'stop_lon']].values)
            d, _ = BallTree(h_coords, metric='haversine').query(pt, k=1)
            dist_lrt = float(d[0][0]) * 6371 * 1000  # metres
        else:
            dist_lrt = 5000.0
        dist_train = dist_lrt * 1.5  # stub

        # Landuse mix (stub: 0–1 entropy-like score)
        landuse_mix = min(1.0, bus_stops_nearby / 10.0)

        rows.append([
            deg, density, road_type,
            bus_stops_nearby, unique_lines, avg_headway, avg_delay,
            mm_starts, mm_ends, bike_len, sim_flow,
            dist_lrt, dist_train, landuse_mix
        ])

    return node_ids, np.array(rows, dtype=np.float32)


print('Building node features...')
node_ids, X_nodes = build_node_features(G_osm)
print(f'Node feature matrix: {X_nodes.shape}')  # (N, 14)

In [ ]:
def build_edge_features(G, node_ids):
    """Returns edge_index (2, E) and edge_attr (E, 8)"""
    id2idx = {nid: i for i, nid in enumerate(node_ids)}
    edges, attrs = [], []

    for u, v, data in G.edges(data=True):
        if u not in id2idx or v not in id2idx:
            continue
        road_len   = _safe(data.get('length', 50))
        travel_t   = _safe(data.get('travel_time', 10))
        speed      = _safe(data.get('speed_kph', 30)) / 130.0  # normalised
        lanes      = _safe(data.get('lanes', 1)) / 6.0
        bike_lane  = 1.0 if data.get('cycleway') not in (None, 'no') else 0.0
        bus_route  = 1.0  # stub: mark all edges as potential bus route
        sim_flow   = road_len * 0.5   # placeholder
        congestion = np.random.uniform(0, 1)  # placeholder

        edges.append([id2idx[u], id2idx[v]])
        attrs.append([road_len, travel_t, speed, lanes, bike_lane, bus_route, sim_flow, congestion])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(attrs, dtype=torch.float)
    return edge_index, edge_attr


print('Building edge features...')
edge_index, edge_attr = build_edge_features(G_osm, node_ids)
print(f'edge_index: {edge_index.shape}  edge_attr: {edge_attr.shape}')  # (2,E), (E,8)

In [ ]:
# Normalise node features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_nodes_norm = scaler.fit_transform(X_nodes).astype(np.float32)

import joblib
joblib.dump(scaler, CFG['out_dir'] / 'node_scaler.pkl')
print('Node features normalised and scaler saved.')

---
## 3. Label Generation

No manual labelling needed — derive labels from the data itself.

| Target | Derivation |
|---|---|
| `accessibility` | reachable nodes within 10 min walk (normalised 0–1) |
| `connectivity` | normalised betweenness centrality |
| `transit_quality` | function of headway + stop density + line count |
| `bike_suitability` | bike lane length + micro-mobility density |
| `network_role` | k-means on centrality + degree |
| `area_type` | rule-based from OSM landuse + transit density |

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

mm = MinMaxScaler()

# ── Regression labels ────────────────────────────────────────────────────────

# accessibility: 10-min walkable nodes (stub: inversely proportional to avg_delay)
y_accessibility = mm.fit_transform(
    (1.0 / (X_nodes[:, 6] + 1)).reshape(-1, 1)  # col 6 = avg_delay
).flatten()

# connectivity: betweenness centrality (expensive on large graphs — use approximation)
print('Computing approximate betweenness centrality (k=200)...')
bc = nx.betweenness_centrality(G_osm, k=200, normalized=True)
y_connectivity = mm.fit_transform(
    np.array([bc.get(n, 0) for n in node_ids]).reshape(-1, 1)
).flatten()

# transit_quality: composite of headway (inverted), stop density, line count
headway_inv = 1.0 / (X_nodes[:, 5] + 1)        # col 5 = avg_headway
stop_dens   = X_nodes[:, 3] / (X_nodes[:, 3].max() + 1e-6)  # col 3 = bus_stops_nearby
line_count  = X_nodes[:, 4] / (X_nodes[:, 4].max() + 1e-6)  # col 4 = unique_lines
y_transit = mm.fit_transform(
    (0.4 * headway_inv + 0.3 * stop_dens + 0.3 * line_count).reshape(-1, 1)
).flatten()

# bike_suitability: bike lane length + micro-mobility
bike_raw = X_nodes[:, 9] + 0.5 * (X_nodes[:, 7] + X_nodes[:, 8])  # cols 9,7,8
y_bike = mm.fit_transform(bike_raw.reshape(-1, 1)).flatten()

# ── Classification labels ────────────────────────────────────────────────────

# network_role: k-means on [degree, betweenness, transit_quality]
role_feats = np.column_stack([X_nodes[:, 0], y_connectivity, y_transit])
kmeans_role = KMeans(n_clusters=CFG['n_role_cls'], random_state=42, n_init=10)
y_role = kmeans_role.fit_predict(role_feats).astype(np.long)

# area_type: rule-based
def classify_area(bus_stops, transit_q, bike_s, degree):
    if transit_q > 0.7:   return 3  # transit_hub
    if bike_s > 0.6:      return 4  # could be industrial — placeholder
    if bus_stops > 5:     return 2  # mixed_use
    if degree > 3:        return 1  # commercial
    return 0                        # residential

y_area = np.array([
    classify_area(X_nodes[i, 3], y_transit[i], y_bike[i], X_nodes[i, 0])
    for i in range(len(node_ids))
], dtype=np.long)

print('Labels:')
for name, arr in [('accessibility', y_accessibility), ('connectivity', y_connectivity),
                  ('transit', y_transit), ('bike', y_bike),
                  ('role', y_role), ('area', y_area)]:
    print(f'  {name:15s}: shape={arr.shape}  range=[{arr.min():.2f}, {arr.max():.2f}]')

---
## 4. Build PyG Data Object

In [ ]:
data = Data(
    x          = torch.tensor(X_nodes_norm,  dtype=torch.float),
    edge_index = edge_index,
    edge_attr  = edge_attr,

    # regression targets
    y_acc      = torch.tensor(y_accessibility, dtype=torch.float).unsqueeze(1),
    y_conn     = torch.tensor(y_connectivity,  dtype=torch.float).unsqueeze(1),
    y_transit  = torch.tensor(y_transit,       dtype=torch.float).unsqueeze(1),
    y_bike     = torch.tensor(y_bike,          dtype=torch.float).unsqueeze(1),

    # classification targets
    y_role     = torch.tensor(y_role, dtype=torch.long),
    y_area     = torch.tensor(y_area, dtype=torch.long),
).to(DEVICE)

print(data)
IN_DIM = data.x.shape[1]
print(f'Input feature dim: {IN_DIM}')

---
## 5. Model Definition — MetroGNN

```
Input features (14)
    ↓
SAGEConv  →  ReLU  →  BatchNorm  →  Dropout
    ↓
SAGEConv  →  ReLU  →  BatchNorm
    ↓
SAGEConv
    ↓
Node embeddings (hidden_dim)
    ↓
global_mean_pool  →  Area embedding vector
    ↓
6 heads
```

In [ ]:
class MetroGNN(torch.nn.Module):

    def __init__(self, in_dim, hidden_dim, n_role_cls, n_area_cls, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.bn1   = BatchNorm(hidden_dim)

        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.bn2   = BatchNorm(hidden_dim)

        self.conv3 = SAGEConv(hidden_dim, hidden_dim)

        # regression heads (output → sigmoid → [0,1])
        self.accessibility_head = Linear(hidden_dim, 1)
        self.connectivity_head  = Linear(hidden_dim, 1)
        self.transit_head       = Linear(hidden_dim, 1)
        self.bike_head          = Linear(hidden_dim, 1)

        # classification heads
        self.role_head = Linear(hidden_dim, n_role_cls)
        self.area_head = Linear(hidden_dim, n_area_cls)

    def encode(self, x, edge_index):
        """Returns per-node embeddings."""
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)

        h = self.conv3(h, edge_index)
        return h  # (N, hidden_dim)

    def forward(self, x, edge_index, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        h = self.encode(x, edge_index)
        g = global_mean_pool(h, batch)   # (B, hidden_dim) — area embedding

        return {
            'node_embeddings': h,
            'area_embedding' : g,
            'accessibility'  : torch.sigmoid(self.accessibility_head(g)),
            'connectivity'   : torch.sigmoid(self.connectivity_head(g)),
            'transit_quality': torch.sigmoid(self.transit_head(g)),
            'bike_suitability': torch.sigmoid(self.bike_head(g)),
            'role_logits'    : self.role_head(g),
            'area_logits'    : self.area_head(g),
        }


model = MetroGNN(
    in_dim      = IN_DIM,
    hidden_dim  = CFG['hidden_dim'],
    n_role_cls  = CFG['n_role_cls'],
    n_area_cls  = CFG['n_area_cls'],
    dropout     = CFG['dropout'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

---
## 6. Training

**Combined loss:**
```
loss = MSE(accessibility) + MSE(connectivity) + MSE(transit) + MSE(bike)
     + CrossEntropy(role) + CrossEntropy(area)
```

For large graphs, use **subgraph mini-batching** (`NeighborLoader`).

In [ ]:
from torch_geometric.loader import NeighborLoader

# For large graphs: sample 2-hop neighborhoods in mini-batches
# This lets you train on a metropolitan graph without OOM.
loader = NeighborLoader(
    data,
    num_neighbors=[15, 10, 5],    # samples per layer (3 layers = 3 entries)
    batch_size=CFG['batch_size'],
    shuffle=True,
)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])


def compute_loss(out, batch_data):
    g = out['area_embedding']
    B = g.size(0)

    # Aggregate batch-level labels (mean over nodes in each subgraph)
    # In NeighborLoader each mini-batch has a single root node → B=batch_size
    # Here we take the label of the root node (index 0 of each subgraph).
    root_mask = torch.zeros(batch_data.num_nodes, dtype=torch.bool)
    root_mask[:B] = True  # NeighborLoader places root nodes first

    loss_acc  = F.mse_loss(out['accessibility'].squeeze(),      batch_data.y_acc[root_mask].squeeze())
    loss_conn = F.mse_loss(out['connectivity'].squeeze(),       batch_data.y_conn[root_mask].squeeze())
    loss_tr   = F.mse_loss(out['transit_quality'].squeeze(),    batch_data.y_transit[root_mask].squeeze())
    loss_bike = F.mse_loss(out['bike_suitability'].squeeze(),   batch_data.y_bike[root_mask].squeeze())
    loss_role = F.cross_entropy(out['role_logits'],             batch_data.y_role[root_mask])
    loss_area = F.cross_entropy(out['area_logits'],             batch_data.y_area[root_mask])

    return loss_acc + loss_conn + loss_tr + loss_bike + loss_role + loss_area, {
        'acc': loss_acc.item(), 'conn': loss_conn.item(),
        'transit': loss_tr.item(), 'bike': loss_bike.item(),
        'role': loss_role.item(), 'area': loss_area.item()
    }


history = []

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss, parts = compute_loss(out, batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    avg = total_loss / len(loader)
    history.append(avg)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{CFG["epochs"]}  loss={avg:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

print('Training complete.')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('MetroGNN Training Loss')
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'training_loss.png', dpi=120)
plt.show()

In [ ]:
# Save model
torch.save(model.state_dict(), CFG['out_dir'] / 'metro_gnn.pt')
print('Model saved to outputs/metro_gnn.pt')

---
## 7. Evaluation

In [ ]:
from sklearn.metrics import (mean_absolute_error, r2_score,
                              classification_report, confusion_matrix)

model.eval()
with torch.no_grad():
    out_full = model(data.x, data.edge_index)

def to_np(t): return t.squeeze().cpu().numpy()

# Regression
for name, pred, true in [
    ('accessibility',   to_np(out_full['accessibility']),   y_accessibility),
    ('connectivity',    to_np(out_full['connectivity']),    y_connectivity),
    ('transit_quality', to_np(out_full['transit_quality']), y_transit),
    ('bike_suitability',to_np(out_full['bike_suitability']),y_bike),
]:
    print(f'{name:18s}  MAE={mean_absolute_error(true, pred):.4f}  R²={r2_score(true, pred):.4f}')

print()

# Classification
role_pred = out_full['role_logits'].argmax(dim=1).cpu().numpy()
area_pred = out_full['area_logits'].argmax(dim=1).cpu().numpy()

print('--- Network Role ---')
print(classification_report(y_role, role_pred, target_names=ROLE_LABELS, zero_division=0))

print('--- Area Type ---')
print(classification_report(y_area, area_pred, target_names=AREA_LABELS, zero_division=0))

---
## 8. Inference — Subgraph Extraction → Structured Output

In [ ]:
id2idx   = {nid: i for i, nid in enumerate(node_ids)}
idx2id   = {i: nid for nid, i in id2idx.items()}


def infer_location(lat: float, lon: float) -> dict:
    """
    Given a GPS coordinate:
    1. Find nearest OSM node.
    2. Extract k-hop subgraph.
    3. Run trained GNN.
    4. Return structured JSON descriptor.
    """
    # Nearest node
    nearest_osm = ox.nearest_nodes(G_osm, X=lon, Y=lat)
    node_idx    = id2idx[nearest_osm]

    # k-hop subgraph
    sub_nodes, sub_edge_index, mapping, _ = k_hop_subgraph(
        node_idx=node_idx,
        num_hops=CFG['k_hops'],
        edge_index=data.edge_index,
        relabel_nodes=True,
        num_nodes=data.num_nodes,
    )

    sub_x = data.x[sub_nodes].to(DEVICE)
    sub_edge_index = sub_edge_index.to(DEVICE)

    model.eval()
    with torch.no_grad():
        out = model(sub_x, sub_edge_index)

    def _f(key): return round(float(out[key].squeeze().cpu()), 3)

    role_idx = int(out['role_logits'].argmax(dim=1).cpu())
    area_idx = int(out['area_logits'].argmax(dim=1).cpu())

    return {
        'lat'              : lat,
        'lon'              : lon,
        'nearest_node'     : int(nearest_osm),
        'subgraph_nodes'   : len(sub_nodes),
        'overall_accessibility': _f('accessibility'),
        'connectivity'     : _f('connectivity'),
        'transit_quality'  : _f('transit_quality'),
        'bike_suitability' : _f('bike_suitability'),
        'network_role'     : ROLE_LABELS[role_idx],
        'area_type'        : AREA_LABELS[area_idx],
    }


# ── Test with exam places ────────────────────────────────────────────────────
exam_df = pd.read_csv('../exam/Places.csv')
results = []
for _, row in exam_df.iterrows():
    res = infer_location(float(row['X']), float(row['Y']))
    res['name'] = row['PLACE']
    res['city'] = row['city']
    results.append(res)
    print(f"{row['PLACE']:30s}  role={res['network_role']:18s}  access={res['overall_accessibility']:.2f}  transit={res['transit_quality']:.2f}")

results_df = pd.DataFrame(results)
results_df.to_csv(CFG['out_dir'] / 'inference_results.csv', index=False, encoding='utf-8-sig')
print('\nSaved inference_results.csv')

---
## 9. LLM Explanation Layer

The LLM does **not** see raw embeddings — it sees structured scores + evidence.

The structured JSON plugs directly into the Map Chat `/api/v1/ask` prompt as extra context.

In [ ]:
def build_llm_context(descriptor: dict) -> str:
    """Convert GNN output to a prompt-ready Hebrew context block."""

    def _score_label(v):
        if v >= 0.75: return 'גבוה'
        if v >= 0.45: return 'בינוני'
        return 'נמוך'

    role_he = {
        'isolated'         : 'מבודד',
        'residential'      : 'שכונתי',
        'local_hub'        : 'צומת מקומי',
        'metropolitan_hub' : 'צומת מטרופוליני',
    }
    area_he = {
        'residential'  : 'מגורים',
        'commercial'   : 'מסחרי',
        'mixed_use'    : 'מעורב',
        'transit_hub'  : 'מוקד תחבורה',
        'industrial'   : 'תעשייה',
    }

    return (
        f"[GNN Transport Context]\n"
        f"רשת: {role_he.get(descriptor['network_role'], descriptor['network_role'])} | "
        f"אזור: {area_he.get(descriptor['area_type'], descriptor['area_type'])}\n"
        f"נגישות: {_score_label(descriptor['overall_accessibility'])} ({descriptor['overall_accessibility']:.2f}) | "
        f"קישוריות: {_score_label(descriptor['connectivity'])} ({descriptor['connectivity']:.2f})\n"
        f"תחבורה: {_score_label(descriptor['transit_quality'])} ({descriptor['transit_quality']:.2f}) | "
        f"אופניים: {_score_label(descriptor['bike_suitability'])} ({descriptor['bike_suitability']:.2f})"
    )


# Example
sample = results[0]
ctx = build_llm_context(sample)
print(f"Place: {sample['name']}\n")
print("Structured JSON:")
print(json.dumps({k: v for k, v in sample.items() if k not in ('lat','lon','nearest_node','subgraph_nodes','name','city')}, ensure_ascii=False, indent=2))
print("\nLLM context block:")
print(ctx)

In [ ]:
# Save full inference results as JSON (ready to inject into LLM prompt)
with open(CFG['out_dir'] / 'inference_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('Saved inference_results.json')

---
## 10. Embedding Visualisation

In [ ]:
# pip install umap-learn
import umap

model.eval()
with torch.no_grad():
    all_emb = model.encode(data.x, data.edge_index).cpu().numpy()

# Sample for speed
N_SAMPLE = min(3000, len(all_emb))
idx_sample = np.random.choice(len(all_emb), N_SAMPLE, replace=False)
emb_sample = all_emb[idx_sample]
role_sample = y_role[idx_sample]

reducer = umap.UMAP(n_components=2, random_state=42)
proj = reducer.fit_transform(emb_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc0 = axes[0].scatter(proj[:, 0], proj[:, 1], c=role_sample, cmap='tab10', s=8, alpha=0.7)
axes[0].set_title('Node embeddings — colored by Network Role')
plt.colorbar(sc0, ax=axes[0])

trans_sample = y_transit[idx_sample]
sc1 = axes[1].scatter(proj[:, 0], proj[:, 1], c=trans_sample, cmap='RdYlGn', s=8, alpha=0.7)
axes[1].set_title('Node embeddings — colored by Transit Quality')
plt.colorbar(sc1, ax=axes[1])

plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'embeddings_umap.png', dpi=120)
plt.show()

---
## Next Steps

- [ ] Replace stub GTFS with real Israel MoT feed (`stops.txt`, `stop_times.txt`)
- [ ] Replace stub SIRI with live delay API
- [ ] Replace stub micro-mobility with real trip data
- [ ] Replace proximity edges with real OSM road topology
- [ ] Add real landuse from OSM polygons (entropy-based `landuse_mix`)
- [ ] Try **GAT** instead of GraphSAGE for attention-weighted neighbours
- [ ] Try **heterogeneous graph** (separate node types: intersection / stop / POI)
- [ ] Temporal extension: SIRI time-series per stop → temporal GNN
- [ ] Wire `build_llm_context()` into `backend/app/services/llm_service.py`